# TP2 — Détection comportementale (UEBA)
## IA non supervisée appliquée à la cybersécurité

**Cours IA & Cybersécurité — Master 1 Ingénierie Aéronautique & Spatiale**

---

### Contexte

Vous êtes analyste SOC (Security Operations Center). Votre SIEM vous remonte des **logs de comportement utilisateurs** sur les 30 derniers jours :
- Heure moyenne de connexion
- Volume de données téléchargées (Mo)
- Nombre d'échecs d'authentification

**Problème** : vous n'avez aucune étiquette — vous ne savez pas à l'avance qui est malveillant.  
**Objectif** : détecter les comportements anormaux **sans supervision**.

---

### Ce que vous allez faire

| Partie | Algorithme | Objectif |
|--------|-----------|---------|
| 1 | **K-Means** | Regrouper les utilisateurs par profil comportemental |
| 2 | **Isolation Forest** | Détecter les outliers — utilisateurs qui ne ressemblent à rien |
| 3 | **Comparaison** | Choisir le bon outil selon le contexte |

---

> 💡 **Rappel** : en IA non supervisée, il n'y a pas de "bonne réponse" connue.  
> L'algorithme cherche de la structure dans les données — c'est à vous d'interpréter.


## Imports et données

Commencez par exécuter cette cellule. Elle charge toutes les librairies nécessaires.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("✓ Librairies chargées")
print("  sklearn :", __import__('sklearn').__version__)
print("  numpy   :", np.__version__)

## Génération des données

Les données simulées représentent **50 utilisateurs** avec 3 features comportementales :

| Feature | Description | Exemple |
|---------|------------|---------|
| `heure_connexion` | Heure moyenne de connexion (0-24h) | 9.2 = 9h12 |
| `volume_Mo` | Volume téléchargé en Mo (30 jours) | 245.0 |
| `nb_echecs_auth` | Nombre d'échecs d'authentification | 2 |

> 📌 Les données sont générées aléatoirement mais reproduisibles grâce à `random_state=42`.


In [ ]:
np.random.seed(42)

# ── Groupe 1 : utilisateurs normaux (bureau, journée) ─────────────────────────
n1 = 30
heure_g1      = np.random.normal(loc=9.5,  scale=1.2, size=n1)   # connexion matin
volume_g1     = np.random.normal(loc=150,  scale=30,  size=n1)   # volume modéré
echecs_g1     = np.random.poisson(lam=1.0, size=n1)              # peu d'échecs

# ── Groupe 2 : utilisateurs nomades (horaires décalés) ───────────────────────
n2 = 15
heure_g2      = np.random.normal(loc=14.0, scale=2.0, size=n2)   # connexion après-midi
volume_g2     = np.random.normal(loc=300,  scale=50,  size=n2)   # volume plus élevé
echecs_g2     = np.random.poisson(lam=2.0, size=n2)

# ── Groupe 3 : comportements suspects (5 utilisateurs) ───────────────────────
n3 = 5
heure_g3      = np.random.uniform(low=0,  high=4,    size=n3)    # connexion nuit
volume_g3     = np.random.uniform(low=800, high=1500, size=n3)   # très gros volume
echecs_g3     = np.random.randint(low=10, high=30,   size=n3)    # beaucoup d'échecs

# ── Assemblage ────────────────────────────────────────────────────────────────
heure   = np.concatenate([heure_g1,  heure_g2,  heure_g3])
volume  = np.concatenate([volume_g1, volume_g2, volume_g3])
echecs  = np.concatenate([echecs_g1, echecs_g2, echecs_g3])

X = np.column_stack([heure, volume, echecs])

print(f"Jeu de données : {X.shape[0]} utilisateurs, {X.shape[1]} features")
print(f"\nAperçu des 5 premiers :")
print(f"{'Utilisateur':>12} {'Heure':>8} {'Volume (Mo)':>12} {'Echecs auth':>12}")
print("-" * 48)
for i in range(5):
    print(f"{'user_'+str(i):>12} {X[i,0]:>8.1f} {X[i,1]:>12.1f} {X[i,2]:>12.0f}")

### Normalisation des données

Avant d'appliquer un algorithme de clustering, il faut **normaliser** les features.

**Pourquoi ?** Les features ont des échelles très différentes :
- `heure_connexion` : entre 0 et 24
- `volume_Mo` : entre 0 et 1500
- `nb_echecs_auth` : entre 0 et 30

Sans normalisation, le volume (grands nombres) dominerait le calcul de distance et les deux autres features seraient ignorées.

> 🔧 `StandardScaler` transforme chaque feature pour qu'elle ait une **moyenne = 0** et un **écart-type = 1**.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Avant normalisation (5 premiers) :")
print(f"  Heure moyenne   : {X[:5,0].mean():.2f}  (min={X[:,0].min():.1f}, max={X[:,0].max():.1f})")
print(f"  Volume moyen    : {X[:5,1].mean():.2f} (min={X[:,1].min():.1f}, max={X[:,1].max():.1f})")
print(f"  Echecs moyens   : {X[:5,2].mean():.2f}  (min={X[:,2].min():.0f}, max={X[:,2].max():.0f})")

print("\nAprès normalisation :")
print(f"  Heure  : moyenne={X_scaled[:,0].mean():.2f}, std={X_scaled[:,0].std():.2f}")
print(f"  Volume : moyenne={X_scaled[:,1].mean():.2f}, std={X_scaled[:,1].std():.2f}")
print(f"  Echecs : moyenne={X_scaled[:,2].mean():.2f}, std={X_scaled[:,2].std():.2f}")

---
## Partie 1 — K-Means : clustering comportemental

### Principe

K-Means regroupe les points en **K clusters** en minimisant la distance entre chaque point et le **centroïde** (centre) de son cluster.

```
1. Placer K centroïdes aléatoirement
2. Assigner chaque point au centroïde le plus proche
3. Recalculer les centroïdes (moyenne des points du cluster)
4. Répéter jusqu'à convergence
```

**Question clé** : comment choisir K ?  
→ La **méthode du coude** : tracer l'inertie (somme des distances²) pour différentes valeurs de K.


### 1.1 — Méthode du coude pour choisir K

In [ ]:
# ── Méthode du coude ─────────────────────────────────────────────────────────
inerties = []
valeurs_k = range(1, 10)

for k in valeurs_k:
    # TODO : créer un KMeans avec k clusters, random_state=42, et l'entraîner sur X_scaled
    # Indice : KMeans(n_clusters=???, random_state=42).fit(???)
    kmeans_tmp = ???
    inerties.append(kmeans_tmp.inertia_)

# Tracé
plt.figure(figsize=(8, 4))
plt.plot(valeurs_k, inerties, 'o-', color='#0077A8', linewidth=2, markersize=8)
plt.xlabel("Nombre de clusters K", fontsize=12)
plt.ylabel("Inertie", fontsize=12)
plt.title("Méthode du coude — Quel K choisir ?", fontsize=13, fontweight='bold')
plt.xticks(valeurs_k)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nObservez le graphe : à quel K l'inertie cesse-t-elle de chuter fortement ?")
print("C'est le 'coude' — la valeur de K optimale.")

**❓ Question 1.1**  
D'après le graphe, quelle valeur de K vous semble optimale ? Justifiez en une phrase.

*Votre réponse :* ___


### 1.2 — Entraînement du modèle K-Means

In [ ]:
# ── Entraînement K-Means avec K=3 ────────────────────────────────────────────
K = 3

# TODO : créer et entraîner un KMeans avec K clusters sur X_scaled
kmeans = ???

# TODO : récupérer les labels (cluster de chaque utilisateur)
# Indice : kmeans.???
labels_kmeans = ???

# TODO : récupérer les centroïdes dans l'espace normalisé
# Indice : kmeans.???
centroides_scaled = ???

print(f"K-Means entraîné avec K={K}")
print(f"Répartition des utilisateurs par cluster :")
for c in range(K):
    n = (labels_kmeans == c).sum()
    print(f"  Cluster {c} : {n} utilisateurs")

### 1.3 — Visualisation des clusters

In [ ]:
couleurs = ['#0077A8', '#F4A61D', '#C0392B']
noms_clusters = ['Cluster 0', 'Cluster 1', 'Cluster 2']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Graphe 1 : Heure de connexion vs Volume téléchargé ───────────────────────
ax = axes[0]
for c in range(K):
    masque = labels_kmeans == c
    ax.scatter(X[masque, 0], X[masque, 1],
               c=couleurs[c], label=noms_clusters[c],
               alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

# TODO : afficher les centroïdes sur le graphe
# Indice : les centroïdes sont dans centroides_scaled (espace normalisé)
# Il faut les repasser dans l'espace original avec scaler.inverse_transform(???)
centroides_orig = ???
ax.scatter(centroides_orig[:, 0], centroides_orig[:, 1],
           c='black', marker='X', s=200, zorder=5, label='Centroïdes')

ax.set_xlabel("Heure de connexion", fontsize=11)
ax.set_ylabel("Volume téléchargé (Mo)", fontsize=11)
ax.set_title("Clusters : Heure vs Volume", fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# ── Graphe 2 : Heure de connexion vs Echecs auth ─────────────────────────────
ax = axes[1]
for c in range(K):
    masque = labels_kmeans == c
    ax.scatter(X[masque, 0], X[masque, 2],
               c=couleurs[c], label=noms_clusters[c],
               alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

ax.scatter(centroides_orig[:, 0], centroides_orig[:, 2],
           c='black', marker='X', s=200, zorder=5, label='Centroïdes')

ax.set_xlabel("Heure de connexion", fontsize=11)
ax.set_ylabel("Nb échecs authentification", fontsize=11)
ax.set_title("Clusters : Heure vs Echecs auth", fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle("K-Means — Clustering comportemental UEBA", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**❓ Question 1.2**  
Observez les clusters. Lequel correspond probablement aux comportements **suspects** ?  
Justifiez en vous appuyant sur les valeurs de centroïde (heure, volume, échecs).

*Votre réponse :* ___

**❓ Question 1.3**  
K-Means a-t-il besoin de savoir à l'avance quels utilisateurs sont malveillants ?  
Que qualifie-t-on d'"apprentissage non supervisé" dans ce contexte ?

*Votre réponse :* ___


---
## Partie 2 — Isolation Forest : détection d'anomalies

### Principe

K-Means regroupe — il cherche des **familles**.  
Isolation Forest isole — il cherche ce qui ne ressemble **à rien**.

**Idée** : un point anormal est plus facile à isoler qu'un point normal.  
L'algorithme construit des arbres de décision aléatoires et mesure combien de coupures sont nécessaires pour isoler chaque point.

```
Score d'anomalie ≈ -1 → anomalie certaine
Score d'anomalie ≈  0 → frontière
Score d'anomalie ≈ +1 → point normal
```

> 📌 Paramètre clé : `contamination` — fraction estimée d'anomalies dans les données.  
> Si vous pensez que ~10% des utilisateurs sont suspects, mettez `contamination=0.1`.


### 2.1 — Entraînement de l'Isolation Forest

In [ ]:
# ── Isolation Forest ─────────────────────────────────────────────────────────

# TODO : créer un IsolationForest avec contamination=0.1 et random_state=42
# Indice : IsolationForest(contamination=???, random_state=42)
iso_forest = ???

# TODO : entraîner le modèle sur X_scaled
???

# TODO : prédire les anomalies (-1 = anomalie, 1 = normal)
# Indice : iso_forest.predict(???)
predictions = ???

# TODO : calculer le score d'anomalie pour chaque utilisateur
# Indice : iso_forest.score_samples(???)
# Note : score négatif = plus suspect
scores = ???

n_anomalies = (predictions == -1).sum()
print(f"Isolation Forest entraîné")
print(f"Utilisateurs détectés comme anomalies : {n_anomalies}")
print(f"\nScores d'anomalie (les 5 plus suspects) :")
indices_tries = np.argsort(scores)  # du plus suspect au moins suspect
for i in indices_tries[:5]:
    statut = "⚠️  ANOMALIE" if predictions[i] == -1 else "   normal  "
    print(f"  user_{i:02d} | score={scores[i]:+.3f} | heure={X[i,0]:.1f}h | vol={X[i,1]:.0f}Mo | echecs={X[i,2]:.0f} | {statut}")

### 2.2 — Visualisation des anomalies détectées

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

normaux   = predictions == 1
anomalies = predictions == -1

# ── Graphe 1 : Heure vs Volume ────────────────────────────────────────────────
ax = axes[0]
ax.scatter(X[normaux, 0], X[normaux, 1],
           c='#0077A8', alpha=0.6, s=60, label='Normal', edgecolors='white', linewidth=0.5)
ax.scatter(X[anomalies, 0], X[anomalies, 1],
           c='#C0392B', alpha=0.9, s=120, marker='X', label='Anomalie détectée', zorder=5)

ax.set_xlabel("Heure de connexion", fontsize=11)
ax.set_ylabel("Volume téléchargé (Mo)", fontsize=11)
ax.set_title("Isolation Forest : Heure vs Volume", fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

# ── Graphe 2 : Heure vs Echecs ────────────────────────────────────────────────
ax = axes[1]
ax.scatter(X[normaux, 0], X[normaux, 2],
           c='#0077A8', alpha=0.6, s=60, label='Normal', edgecolors='white', linewidth=0.5)
ax.scatter(X[anomalies, 0], X[anomalies, 2],
           c='#C0392B', alpha=0.9, s=120, marker='X', label='Anomalie détectée', zorder=5)

ax.set_xlabel("Heure de connexion", fontsize=11)
ax.set_ylabel("Nb échecs authentification", fontsize=11)
ax.set_title("Isolation Forest : Heure vs Echecs auth", fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle("Isolation Forest — Détection d'anomalies UEBA", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nUtilisateurs flaggés comme anomalies :")
for i in np.where(anomalies)[0]:
    print(f"  user_{i:02d} | heure={X[i,0]:.1f}h | volume={X[i,1]:.0f}Mo | echecs={X[i,2]:.0f} | score={scores[i]:+.3f}")

**❓ Question 2.1**  
Comparez les utilisateurs flaggés par Isolation Forest avec le cluster suspect identifié en Partie 1.  
Les résultats sont-ils identiques ? Qu'est-ce qui explique les différences éventuelles ?

*Votre réponse :* ___

**❓ Question 2.2**  
Que se passe-t-il si vous changez `contamination=0.05` (5%) au lieu de `contamination=0.1` (10%) ?  
Essayez et observez. Quel est l'impact sur les faux positifs ?

*Votre réponse :* ___


### 2.3 — Bonus : visualisation des scores d'anomalie

Cette cellule est déjà complète. Exécutez-la et interprétez le graphe.


In [ ]:
plt.figure(figsize=(12, 4))

couleurs_bar = ['#C0392B' if p == -1 else '#0077A8' for p in predictions]
indices = np.arange(len(scores))

plt.bar(indices, -scores, color=couleurs_bar, alpha=0.8, edgecolor='white', linewidth=0.3)
plt.axhline(y=-scores[predictions == -1].max() * 0.95,
            color='#C0392B', linestyle='--', linewidth=1.5, label='Seuil anomalie', alpha=0.7)

patch_normal   = mpatches.Patch(color='#0077A8', label='Normal')
patch_anomalie = mpatches.Patch(color='#C0392B', label='Anomalie')
plt.legend(handles=[patch_normal, patch_anomalie], fontsize=11)

plt.xlabel("Utilisateur (index)", fontsize=11)
plt.ylabel("Score d'anomalie (plus haut = plus suspect)", fontsize=11)
plt.title("Scores d'anomalie — Isolation Forest", fontsize=13, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Les barres rouges sont les utilisateurs détectés comme anomalies.")
print("Plus la barre est haute, plus l'utilisateur est 'facile à isoler'.")

---
## Partie 3 — Comparaison & réflexion

### 3.1 — Tableau comparatif


In [ ]:
# Tableau comparatif — exécutez cette cellule
print(f"{'Critère':<28} {'K-Means':<25} {'Isolation Forest':<25}")
print("─" * 78)
comparaison = [
    ("Type",                  "Clustering",              "Détection d'anomalies"),
    ("Ce qu'il cherche",      "Des familles similaires", "Des points isolés"),
    ("Paramètre principal",   "K (nb de clusters)",      "contamination (% suspects)"),
    ("Résultat",              "Label de cluster (0,1,2)","Normal (+1) / Anomalie (-1)"),
    ("Interprétation",        "Manuelle (analyste SOC)", "Automatique (seuil)"),
    ("Bon pour",              "Profilage comportemental","Détection ponctuelle"),
    ("Limite principale",     "Doit choisir K",          "Sensible à contamination"),
    ("Outil SIEM équivalent", "Splunk ML clustering",    "Microsoft Sentinel Anomaly"),
]
for c, k, iso in comparaison:
    print(f"  {c:<26} {k:<25} {iso:<25}")

### 3.2 — Questions de réflexion

**❓ Question 3.1 — En production**  
Dans un SOC réel, lequel utiliseriez-vous en premier ? Pourquoi ?  
*(Pensez au nombre d'alertes générées, à la charge de l'analyste, à la criticité du SI)*

*Votre réponse :* ___

**❓ Question 3.2 — Limites communes**  
Les deux algorithmes sont **non supervisés** : ils ne savent pas ce qu'est une vraie attaque.  
Quel risque cela représente-t-il en opérationnel ?  
Comment les SIEM modernes (Splunk, Sentinel, Darktrace) contournent-ils ce problème ?

*Votre réponse :* ___

**❓ Question 3.3 — Lien avec l'IA symbolique (TP_S1)**  
Dans le TP_S1, vous utilisiez des règles SI→ALORS pour détecter les intrusions.  
Ici, vous n'avez écrit aucune règle.  
Listez 2 avantages et 2 inconvénients de l'approche non supervisée par rapport aux règles.

| | Règles (TP_S1) | Non supervisé (TP2) |
|--|---|---|
| ✓ Avantage 1 | | |
| ✓ Avantage 2 | | |
| ✗ Inconvénient 1 | | |
| ✗ Inconvénient 2 | | |


---
## 🎯 Exercice bonus — Combinaison K-Means + Isolation Forest

En pratique, les deux algorithmes sont souvent combinés :
1. K-Means identifie les clusters comportementaux
2. Isolation Forest détecte les outliers **à l'intérieur de chaque cluster**

Complétez le code suivant pour appliquer Isolation Forest **séparément sur chaque cluster**.


In [ ]:
# BONUS : Isolation Forest par cluster
fig, axes = plt.subplots(1, K, figsize=(15, 5))

for c in range(K):
    ax = axes[c]
    
    # TODO : sélectionner les données du cluster c
    # Indice : X_scaled[labels_kmeans == ???]
    X_cluster = ???
    indices_cluster = np.where(labels_kmeans == c)[0]
    
    if len(X_cluster) < 5:
        ax.set_title(f"Cluster {c} (trop petit)", fontsize=11)
        continue
    
    # TODO : entraîner un Isolation Forest sur ce cluster uniquement
    # Utilisez contamination=0.15, random_state=42
    iso_cluster = ???
    ???
    
    # TODO : prédire les anomalies dans ce cluster
    pred_cluster = ???
    
    anomalies_c = pred_cluster == -1
    normaux_c   = pred_cluster == 1
    
    # Visualisation heure vs volume pour ce cluster
    X_orig_cluster = X[indices_cluster]
    ax.scatter(X_orig_cluster[normaux_c, 0], X_orig_cluster[normaux_c, 1],
               c='#0077A8', alpha=0.7, s=60, label='Normal')
    if anomalies_c.sum() > 0:
        ax.scatter(X_orig_cluster[anomalies_c, 0], X_orig_cluster[anomalies_c, 1],
                   c='#C0392B', s=120, marker='X', label='Anomalie', zorder=5)
    
    ax.set_title(f"Cluster {c} — {anomalies_c.sum()} anomalie(s)", fontsize=11, fontweight='bold')
    ax.set_xlabel("Heure connexion")
    ax.set_ylabel("Volume (Mo)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("Bonus : Isolation Forest par cluster K-Means", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Récapitulatif

| Concept | Ce que vous avez fait |
|---------|----------------------|
| **Normalisation** | StandardScaler pour mettre les features à la même échelle |
| **Méthode du coude** | Choisir K en traçant l'inertie |
| **K-Means** | Regrouper 50 utilisateurs en 3 profils comportementaux |
| **Isolation Forest** | Détecter automatiquement les utilisateurs anormaux |
| **Comparaison** | Comprendre quand utiliser l'un vs l'autre |

**Prochain TP** : Deep Learning — réseaux de neurones pour la détection d'intrusion.
